# Sesión 7 — K-means y DBSCAN: encontrar grupos sin respuesta

**Introducción al Análisis Avanzado de Datos con Python** · Univalle · Módulo 3 — No supervisados

## Objetivos de hoy
Al terminar la sesión usted podrá:
1. Explicar qué hace K-means por dentro (asignar → promediar) y qué minimiza (la **inercia**), y por qué depende del punto de partida.
2. Elegir un número de grupos `k` con el **codo** y la **silueta**, sabiendo que son pistas y no veredictos.
3. Usar **DBSCAN** cuando los grupos no son redondos o hay ruido, y explicar sus perillas `eps` y `min_samples`.
4. Detectar cuándo un clustering está agrupando **por unidad de medida** y no por lo que importa, y corregirlo escalando.

## Dónde estamos en el pipeline

```
 ┌────────┐   ┌──────────────────┐   ┌────────┐   ┌────────────┐   ┌────────────────┐
 │ DATOS  │ → │ PREPROCESAMIENTO │ → │ MODELO │ → │ EVALUACIÓN │ → │ INTERPRETACIÓN │
 └────────┘   └──────────────────┘   └────────┘   └────────────┘   └────────────────┘
                                       ▲▲▲▲▲▲▲▲                      ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲
   HOY: modelos SIN y. No hay respuesta correcta que imitar: hay que buscar estructura… y saber leerla.
```

🔍 **Predice** · ✏️ **Completa** (andamiaje medio) · 💥 **Trampa** · 🖼️ **Fichas** · ⚡ **Mini-reto**.

> **Datos:** primero datos de laboratorio (`make_blobs`, `make_moons`: sabemos la verdad, el algoritmo no) y después **Palmer Penguins**: 344 pingüinos de la Antártida medidos por la Estación Palmer (Gorman, Williams y Fraser, 2014; Horst, Hill y Gorman, 2020; CC0).


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

pd.set_option("display.max_columns", 40)
REPO = "https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/"
ARCHIVOS = {"penguins.csv": "datos/penguins.csv", "figuras.py": "tools/figuras.py"}

def asegurar_datos(archivo, url_repo):
    """Deja un archivo en el disco de Colab: lo baja del repositorio del curso si aún no está."""
    from pathlib import Path
    import requests
    if Path(archivo).exists():
        return True
    try:
        r = requests.get(url_repo, timeout=120)
        if r.ok:
            Path(archivo).write_bytes(r.content)
            return True
    except Exception:
        pass
    print(f"⚠️ No pude bajar {archivo}. Use el Plan B (celda siguiente).")
    return False

for archivo, ruta in ARCHIVOS.items():
    asegurar_datos(archivo, REPO + ruta)
import figuras as fg
print("Listo:", ", ".join(ARCHIVOS))

<details><summary><b>Plan B</b> — si la celda anterior avisó que no pudo bajar algún archivo</summary>

La celda siguiente revisa sola qué falta y solo en ese caso le pide subirlo.
</details>

In [ ]:
# Plan B: solo actúa si falta algún archivo. Con "Ejecutar todo" no se detiene.
from pathlib import Path
faltan = [a for a in ARCHIVOS if not Path(a).exists()]
if not faltan:
    print("✓ Todo disponible: no hace falta el Plan B.")
else:
    from google.colab import files
    print("⚠️ Faltan:", faltan, "→ súbalos")
    files.upload()

In [ ]:
fg.pipeline(["modelo", "interpretación"], modulo=3, subtitulo="HOY: modelos sin y — encontrar grupos y saber leerlos");

## 1. Se acabó la *y*

En los módulos 1 y 2 cada fila traía su respuesta (¿hubo víctimas?, ¿cuánto cobra el seguro?, ¿abrió el depósito?) y el modelo aprendía a imitarla. En el **aprendizaje no supervisado** no hay respuesta: solo las *X*. La pregunta cambia de *"¿qué va a pasar?"* a *"¿qué estructura tienen estos datos?"*.

In [ ]:
fg.supervisado_vs_no_supervisado();

**Clustering** (agrupamiento) es la tarea no supervisada más común: partir las filas en grupos de filas **parecidas entre sí** y **distintas de las de otros grupos**. Se usa para segmentar clientes, agrupar municipios con realidades parecidas, encontrar comportamientos raros (fraude, sensores dañados) o simplemente para entender un dataset nuevo.

Empezamos con datos de **laboratorio**: `make_blobs` fabrica nubes de puntos alrededor de centros que nosotros elegimos. Nosotros sabemos la verdad; el algoritmo no la va a ver.

In [ ]:
from sklearn.datasets import make_blobs

X_lab, grupo_verdadero = make_blobs(n_samples=500, centers=[(-5, -2), (-1, 5), (2, -1), (4.5, 1.5)],
                                    cluster_std=[1.2, 1.3, 1.0, 1.0], random_state=7)
X_lab = pd.DataFrame(X_lab, columns=["x1", "x2"])
X_lab.head()

### 🔍 Predice
Mire la figura de abajo **sin** pensar en cuántos centros usamos. ¿Cuántos grupos ve usted?

**Tu predicción:** ____ grupos

In [ ]:
fg.puntos_sin_color(X_lab);

## 2. K-means: asignar y promediar

K-means necesita que le digamos **cuántos grupos** buscar (`k`). Después repite dos pasos:

1. **Asignar:** cada punto se va con el **centroide** (el centro del grupo) más cercano.
2. **Promediar:** cada centroide se muda al **promedio** de los puntos que le tocaron.

…hasta que los centroides ya no se mueven. Los primeros centroides se escogen al azar.

> 🧪 **Laboratorio** — antes de correr el código, véalo funcionando: [**K-means**](https://nitobest.github.io/curso-aad-python/laboratorio/kmeans.html) · asignar y mover, paso a paso; el reto de los pingüinos con y sin escalar. Tres pestañas: **Mira**, **Juega** y **Reto** (prediga antes de correr). Funciona en el celular.

In [ ]:
fg.kmeans_iteraciones(X_lab, k=4, semilla=8, mostrar=(0, 1, 2, 4, -1));

### 📐 Fundamento
Lo que K-means intenta hacer pequeño es la **inercia**: la suma, sobre todos los puntos, de la distancia al cuadrado a su centroide.

$$\text{inercia} = \sum_{i=1}^{n} \lVert x_i - \mu_{c(i)} \rVert^2$$

- $x_i$ es el punto $i$; $\mu_{c(i)}$ es el centroide del grupo al que quedó asignado.
- Inercia **baja** = puntos apretados alrededor de su centro.

Es la misma idea de los **residuales al cuadrado** de la regresión (Sesión 4): allí cada punto se comparaba con la recta; aquí, con su centroide.

<details><summary><b>Para quien quiera más:</b> por qué la inercia nunca sube, y por qué igual puede quedar mal</summary>

El paso *asignar* no puede subir la inercia (cada punto se va al centro más cercano, o se queda). El paso *promediar* tampoco: el promedio es justamente el punto que minimiza la suma de distancias² a un grupo de puntos. Por eso el algoritmo siempre termina. Pero termina en un **mínimo local**: el mejor arreglo *cerca* de donde arrancó, no necesariamente el mejor posible. Encontrar el mínimo global es un problema computacionalmente muy difícil (NP-difícil), así que en la práctica se arranca varias veces (`n_init`) y se escogen arranques inteligentes (**k-means++**: el primer centroide al azar y cada siguiente lejos de los anteriores).
</details>

![Dentro del .fit() — K-means y DBSCAN](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S07/img/dentro_del_fit_kmeans_dbscan.png)

### ✏️ Completa — entrene K-means
Con `k = 4`, 10 arranques (`n_init=10`) y `random_state=42`. Ojo: **no hay `y`** en el `.fit()`.

In [ ]:
from sklearn.cluster import KMeans

kmeans = ...   # TODO: KMeans(n_clusters=4, n_init=10, random_state=42) entrenado con X_lab (sin y)
print("grupo de las primeras 10 filas:", kmeans.labels_[:10])
print("inercia:", round(kmeans.inertia_, 1))
pd.DataFrame(kmeans.cluster_centers_, columns=["x1", "x2"]).round(2)

`labels_` es el grupo de cada fila (0, 1, 2, 3: los números son **nombres**, no un orden) y `cluster_centers_` son los centroides. Eso es **todo** lo que aprende K-means: k puntos.

In [ ]:
fg.kmeans_resultado(X_lab, kmeans);

Como aprendió centroides, K-means puede ubicar filas **nuevas**: `.predict()` mide la distancia a cada centroide y escoge la más corta.

In [ ]:
nuevos = pd.DataFrame({"x1": [0, 4], "x2": [6, 0]})
nuevos.assign(grupo=kmeans.predict(nuevos))

## 3. ¿Cuántos grupos? La perilla `k`

### 🔍 Predice
Si le pedimos a K-means **6** grupos sobre los mismos datos, ¿qué cree que hace? ¿Se niega, deja grupos vacíos, o…?

**Tu predicción:** ____

In [ ]:
fg.kmeans_k_efecto(X_lab, ks=(2, 3, 4, 6));

In [ ]:
fg.codo_silueta(X_lab, marcar=4);

### La semilla importa
¿Qué pasa si K-means arranca **una sola vez** (`n_init=1`) desde centroides al azar? Mismo `k = 4`, cuatro semillas distintas:

In [ ]:
fg.kmeans_semillas(X_lab, k=4, semillas=(0, 1, 2, 3));

Con las semillas 0 y 2, K-means se queda atascado en un **mínimo local**: dos centroides se pelean una nube y otra nube queda partida. La inercia lo delata (≈1 900 contra ≈1 160). `n_init=10` corre diez arranques y se queda con el de menor inercia; por eso lo usamos siempre. Guarde esta idea para la Sesión 8: **un resultado que cambia con la semilla no es un hallazgo.**

## 4. Cuando los grupos no son redondos: DBSCAN

K-means supone que cada grupo es una nube alrededor de un centro. ¿Y si los grupos tienen otra forma? `make_moons` fabrica dos lunas entrelazadas; les agregamos 25 puntos sueltos (ruido).

In [ ]:
from sklearn.datasets import make_moons

lunas, _ = make_moons(n_samples=400, noise=0.07, random_state=0)
sueltos = np.random.default_rng(1).uniform([-1.3, -0.8], [2.3, 1.3], size=(25, 2))
X_lunas = pd.DataFrame(np.vstack([lunas, sueltos]), columns=["x1", "x2"])
fg.kmeans_vs_dbscan(X_lunas, k=2, eps=0.15);

K-means parte con una recta, porque solo sabe de centros. **DBSCAN** (*Density-Based Spatial Clustering of Applications with Noise*) piensa distinto: un grupo es una **zona densa** de puntos, tenga la forma que tenga.

In [ ]:
fg.dbscan_idea(X_lunas, eps=0.15, min_samples=5);

### 📐 Fundamento
DBSCAN tiene dos perillas: un radio $\varepsilon$ (`eps`) y un mínimo de vecinos (`min_samples`).

- **Núcleo:** punto con al menos `min_samples` puntos (contándose a sí mismo) a distancia $\le \varepsilon$.
- **Borde:** punto que no es núcleo pero está a $\le \varepsilon$ de algún núcleo.
- **Ruido:** todo lo demás. DBSCAN le pone la etiqueta **−1**.

Un cluster es un conjunto de núcleos **encadenados** (cada uno a $\le \varepsilon$ del siguiente) más sus bordes. No hay $k$ ni función de pérdida: el número de grupos **sale** de los datos.

### ✏️ Completa — DBSCAN sobre las lunas
Use `eps=0.15` y `min_samples=5`. DBSCAN no tiene `.predict()`; se usa `.fit_predict()`, que entrena y devuelve las etiquetas de las mismas filas.

In [ ]:
from sklearn.cluster import DBSCAN

etiquetas_db = ...   # TODO: DBSCAN(eps=0.15, min_samples=5) con .fit_predict(X_lunas)
pd.Series(etiquetas_db).value_counts().rename("puntos").rename_axis("etiqueta (−1 = ruido)")

### 🔍 Predice
Con `eps = 0.3` (el doble), ¿cuántos grupos encontrará? ¿Y con `eps = 0.05`?

**Tu predicción:** con 0,3 → ____ · con 0,05 → ____

In [ ]:
fg.dbscan_eps(X_lunas, epsilons=(0.05, 0.1, 0.15, 0.3));

![¿K-means o DBSCAN?](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S07/img/kmeans_o_dbscan.png)

## 5. Datos reales: 342 pingüinos

En la Estación Palmer (Antártida) midieron pingüinos de **tres especies**: Adelia, barbijo (*Chinstrap*) y papúa (*Gentoo*). Tenemos cuatro medidas por pingüino: largo y alto del pico (mm), largo de la aleta (mm) y masa corporal (g).

Vamos a **esconder la especie**: agrupamos solo con las cuatro medidas y al final miramos si los grupos coinciden con las especies. La especie es nuestra "verdad de laboratorio" en datos reales.

In [ ]:
pinguinos = pd.read_csv("penguins.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
pinguinos = pinguinos.dropna(subset=medidas).reset_index(drop=True)
print(pinguinos.shape)
pinguinos[medidas].describe().round(1)

### 🔍 Predice
Le pedimos a K-means `k = 3` grupos usando las cuatro medidas **tal como vienen**. Hay tres especies. ¿Qué porcentaje de pingüinos cree que quedará en el grupo de "su" especie?

**Tu predicción:** ____ %

### ✏️ Completa — K-means con las medidas tal como vienen

In [ ]:
km_crudo = ...   # TODO: KMeans(n_clusters=3, n_init=10, random_state=42) con pinguinos[medidas]
fg.tabla_cruzada(pinguinos["species"], km_crudo.labels_);

**68 %.** Gentoo queda partido en dos grupos y Adelia y barbijo quedan revueltos. ¿Las medidas no sirven para distinguir especies? Miremos cómo partió K-means a los pingüinos:

In [ ]:
pinguinos.assign(cluster=km_crudo.labels_).groupby("cluster")["body_mass_g"].agg(["min", "max", "mean"]).round(0)

## 6. 💥 Agrupó por gramos

Los tres grupos son **tres franjas de peso**: menos de 3 975 g, de 4 000 a 4 950 g, y más de 4 975 g. Las otras tres medidas casi no participaron.

### ✏️ Completa — escale y repita
`StandardScaler` (Sesión 2) deja cada medida con promedio 0 y desviación 1. No hay *y* ni prueba aquí: se ajusta con todos los pingüinos.

In [ ]:
from sklearn.preprocessing import StandardScaler

escalador = StandardScaler()
Z = escalador.fit_transform(pinguinos[medidas])
km_z = ...   # TODO: el mismo KMeans, ahora con Z
fg.tabla_cruzada(pinguinos["species"], km_z.labels_);

In [ ]:
fg.clusters_por_unidad(pinguinos, "body_mass_g", "bill_length_mm", km_crudo.labels_, km_z.labels_);

## 7. Una fila entra, un grupo sale

El primer pingüino de la tabla es un Adelia de 3 750 g. ¿Por qué K-means lo pone donde lo pone? Su distancia² a cada centroide es la **suma** de las diferencias² en cada medida: los colores muestran cuánto aporta cada una.

In [ ]:
fila = pinguinos.loc[0, medidas]
fg.kmeans_una_fila([("sin escalar (gramos y milímetros)", km_crudo.cluster_centers_, fila.values),
                    ("escalado (z)", km_z.cluster_centers_, escalador.transform(fila.to_frame().T)[0])], medidas);

Sin escalar, la barra es casi toda **naranja** (masa corporal): la decisión la toma una sola variable. Escalado, las cuatro medidas aportan y el pingüino cae en el grupo de los Adelia.

### ¿Cómo es cada grupo?
Los centroides están en unidades *z*. `inverse_transform` los devuelve a milímetros y gramos para poder **describir** cada grupo (en la Sesión 9 hacemos esto en serio):

In [ ]:
perfil = pd.DataFrame(escalador.inverse_transform(km_z.cluster_centers_), columns=medidas).round(1)
perfil["pingüinos"] = np.bincount(km_z.labels_)
perfil["especie más común"] = pinguinos.groupby(km_z.labels_)["species"].agg(lambda s: s.mode()[0])
perfil

### 🔍 Predice
Si no supiéramos que hay 3 especies, ¿qué `k` recomendaría la silueta con los pingüinos escalados?

**Tu predicción:** k = ____

In [ ]:
fg.codo_silueta(Z, ks=range(1, 8), marcar=3);

## ⚡ Mini-reto (opcional)

¿DBSCAN encuentra las tres especies? Pruébelo sobre `Z` (escalado) con varios `eps` entre 0,3 y 0,9 y `min_samples=5`. Para cada uno, cuente grupos y ruido y mire la tabla cruzada. ¿Qué pasa con 0,3? ¿Por qué ningún `eps` separa bien a Adelia de barbijo? (Pista: la ficha "¿K-means o DBSCAN?", la línea de densidades.)

In [ ]:
# Su análisis aquí
# for eps in (0.3, 0.5, 0.7, 0.9):
#     etiquetas = DBSCAN(eps=eps, min_samples=5).fit_predict(Z)
#     ...

## 8. Cierre — qué cambió hoy en el pipeline

| Etapa | Hoy (Sesión 7) |
|---|---|
| Datos | sin *y*; la etiqueta conocida (especie) se esconde y solo se usa al final para **leer** los grupos |
| Preprocesamiento | **escalar** deja de ser opcional: sin escalar, el clustering agrupa por unidades |
| Modelo | K-means (centroides, `k`, `n_init`) y DBSCAN (densidad, `eps`, `min_samples`) |
| Evaluación | no hay accuracy: inercia (codo), silueta y comparación con algo externo |
| Interpretación | perfil de cada grupo en unidades originales; un grupo que cambia con la semilla no es un hallazgo |

**Dentro del `.fit()` — el curso hasta hoy:**

| | Árbol | Lineal | Logística | k-NN | Bosque | K-means | DBSCAN |
|---|---|---|---|---|---|---|---|
| ¿Usa *y*? | sí | sí | sí | sí | sí | **no** | **no** |
| Qué minimiza | Gini | residuales² | log-loss | nada | Gini (por árbol) | inercia | nada |
| Cómo lo busca | voraz | cerrada o gradiente | iterativo | guarda y vota | B árboles | iterativo (Lloyd), mínimo local | encadena vecindarios |
| Perilla del modelo | `max_depth` | `alpha` | `C` | `k` | `n_estimators` | `k` (`n_clusters`) | `eps`, `min_samples` |
| ¿Escalar? | no | a veces | sí | **sí** | no | **sí** | **sí** |

**Para la Sesión 8:** hoy dibujamos los pingüinos con 2 de sus 4 medidas. ¿Cómo se "ve" un dataset de 4, 15 o 100 variables? **PCA** y **t-SNE**: reducir dimensiones… y la trampa de creerle todo a un dibujo bonito.